# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samin-developer/ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. The Data Contract (In Plain Words)

### Answer 1: What one row means for my lane (Lane 1 - Relevance)

**One row = one query-page pair** with an associated relevance label (0/1), along with features about the query, the page, and their interaction.

Each row represents a single search result that was shown to a user, with information about:
- The search query the user entered
- The page that was displayed as a result
- Whether the user found that page relevant

---

### Answer 2: Which table(s) I'll use

**Table:** `internship_warehouse.lane1_relevance` (or similar)

**Why:** This table contains the query-page pairs with relevance labels, along with the features I need.

**Alternate tables:** If needed, I may also use:
- `internship_warehouse.search_events` - for user interaction data
- `internship_warehouse.page_metadata` - for page-level features

---

### Answer 3: Which time window

**Time window:** March 2026 (`month=2026-03`)

**Why March?**
- It's a mid-panel month (not the final month, which is June 2026)
- It allows me to develop and test features without using the final evaluation month
- The _sample table is June 2026 (final month) - I'll use that only for final testing

**Date range:** March 1, 2026 to March 31, 2026

---

### Answer 4: What I'd predict or rank (label or proxy)

**Label:** `relevance` (binary: 1 = relevant, 0 = not relevant)

**Proxy (if label unavailable):** `click_through_rate` (CTR) as a proxy for relevance

**Why this label:**
- It's the direct measure of whether a page answers the user's question
- It's the target variable from the starter notebook
- It's what my model will learn to predict

---

### Answer 5: One thing I deliberately exclude

**What I exclude:** `position` in the final ranking decision

**Why:** Position is a result of the ranking, not a cause of relevance. Including position would be a classic "leakage" trap because:
- A page's position is determined by the ranking algorithm
- Position influences CTR, so it would look like a strong predictor
- But it's not available when we need to make a ranking decision (we decide the position)

**What I'll use instead:** Features that are knowable before ranking, like:
- Query-page text similarity
- Page freshness
- Page authority

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [4]:
# ================================================
# CELL 1: Setup Hugging Face Access
# ================================================

import os
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt

# Check if running in Colab
try:
    from google.colab import userdata
    IN_COLAB = True
except:
    IN_COLAB = False

# Set up Hugging Face token
if IN_COLAB:
    # Use Colab Secrets (don't paste token directly)
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
        print("✅ HF_TOKEN loaded from Colab Secrets")
    except:
        print("⚠️ HF_TOKEN not found in Colab Secrets")
        print("   Please add HF_TOKEN to Secrets (🔑 key icon)")
        print("   Or manually set it below (not recommended for public repos)")
        HF_TOKEN = None
else:
    # Local environment - set environment variable
    HF_TOKEN = os.environ.get('HF_TOKEN')
    if HF_TOKEN:
        print("✅ HF_TOKEN loaded from environment")
    else:
        print("⚠️ HF_TOKEN not found in environment")

# Load dataset from Hugging Face
from huggingface_hub import hf_hub_download
import json

# Define dataset and file
DATASET_REPO = "FlyRank/internship-warehouse"
FILE_PATH = "data/warehouse_sample.json"  # This may need to be adjusted

# Download the dataset
if HF_TOKEN:
    try:
        # Download the file
        local_path = hf_hub_download(
            repo_id=DATASET_REPO,
            filename=FILE_PATH,
            token=HF_TOKEN
        )
        print(f"✅ Downloaded: {local_path}")

        # Load the data
        with open(local_path, 'r') as f:
            data = json.load(f)
        df = pd.DataFrame(data)
        print(f"✅ Loaded {len(df):,} rows")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"❌ Error: {e}")
        print("   Please check:")
        print("   1. HF_TOKEN is set correctly")
        print("   2. You have access to the dataset")
        print("   3. The file path is correct")
else:
    print("❌ No HF_TOKEN found - cannot access dataset")
    print("   Please add HF_TOKEN to Colab Secrets or environment")

✅ HF_TOKEN loaded from Colab Secrets
❌ Error: 404 Client Error. (Request ID: Root=1-6a56530f-354f352a116af14c2b454e08;688d861e-3f28-4011-8f19-d142aae551bc)

Repository Not Found for url: https://huggingface.co/FlyRank/internship-warehouse/resolve/main/data/warehouse_sample.json.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
   Please check:
   1. HF_TOKEN is set correctly
   2. You have access to the dataset
   3. The file path is correct


In [5]:
from huggingface_hub import hf_hub_download

# Try downloading a known public file to test your setup
try:
    test_path = hf_hub_download(
        repo_id="huggingface/transformers",
        filename="README.md",
        token=HF_TOKEN  # Your token is still valid
    )
    print(f"✅ Test download successful! File saved at: {test_path}")
except Exception as e:
    print(f"❌ Test download failed. There might be an issue with your token or internet connection: {e}")

❌ Test download failed. There might be an issue with your token or internet connection: 404 Client Error. (Request ID: Root=1-6a565394-3bb465e42ed4cbb042103a98;ade24aa6-43f2-4b8b-93ea-1017e259d17a)

Repository Not Found for url: https://huggingface.co/huggingface/transformers/resolve/main/README.md.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
# ================================================
# CELL 2: Query 1 - Verify the Grain
# ================================================

# This query verifies that one row = one query-page pair

print("=" * 50)
print("QUERY 1: VERIFY THE GRAIN")
print("=" * 50)
print("Goal: Confirm that one row represents one query-page pair")

# Check for duplicates - each (query, page) pair should be unique
if 'query' in df.columns and 'page' in df.columns:
    duplicates = df.groupby(['query', 'page']).size().reset_index(name='count')
    duplicates = duplicates[duplicates['count'] > 1]

    print(f"Total rows: {len(df):,}")
    print(f"Unique (query, page) pairs: {len(df.groupby(['query', 'page'])):,}")
    print(f"Duplicate pairs: {len(duplicates)}")

    if len(duplicates) == 0:
        print("✅ One row = one query-page pair (no duplicates)")
    else:
        print(f"⚠️ Found {len(duplicates)} duplicate pairs")
else:
    print("⚠️ 'query' and 'page' columns not found")
    print(f"Available columns: {df.columns.tolist()}")

QUERY 1: VERIFY THE GRAIN
Goal: Confirm that one row represents one query-page pair


NameError: name 'df' is not defined

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.